# Notebook 5 – ReAct Pattern with Gemini
## Reasoning + Acting Loop for an Industry Agent

---

### What you will learn

- What the **ReAct pattern** is
- Why it is different from Reactive agents
- How Gemini is used inside a **reason → act → observe loop**
- How tools and LLMs cooperate safely

Use case (unchanged):
> **E-commerce Customer Support Agent**

## 1. Why ReAct?

ReAct introduces an explicit loop:

Thought → Action → Observation

This allows agents to adapt dynamically.

In [1]:
def fetch_order(order_id: str):
    if order_id == "123":
        return "Order delayed by 3 days"
    return "Order not found"

def offer_coupon():
    return "10% discount coupon issued"

In [2]:
import os, re, json
import google.generativeai as genai

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-2.5-flash")

def safe_json_parse(text: str):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError("No JSON found")
    return json.loads(match.group())

def gemini_react_think(context: str):
    prompt = f"""
        Return ONLY valid JSON.
        {{"thought": string, "action": string, "action_input": string}}

        Context:
        {context}

        Allowed actions: fetch_order, offer_coupon, respond_to_user
        """
    response = model.generate_content(prompt)
    return safe_json_parse(response.text)

c:\Users\Lucifer\anaconda3\envs\rag101\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\Users\Lucifer\anaconda3\envs\rag101\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class ReActAgent:
    def __init__(self, max_steps=3):
        self.max_steps = max_steps

    def run(self, user_message: str):
        context = f"User: {user_message}"

        for step in range(self.max_steps):
            state = gemini_react_think(context)
            thought = state["thought"]
            action = state["action"]
            action_input = state.get("action_input", "")

            print(f"STEP {step+1} THOUGHT:", thought)

            if action == "fetch_order":
                obs = fetch_order(action_input)
            elif action == "offer_coupon":
                obs = offer_coupon()
            elif action == "respond_to_user":
                return action_input
            else:
                obs = "Unknown action"

            print("OBSERVATION:", obs)
            context += f"\nObservation: {obs}"

In [4]:
agent = ReActAgent()
final = agent.run("My order 123 has not arrived and I am angry. If I do not recieve it by tomorrow can I get a discount coupon?")
print("FINAL RESPONSE:", final)

STEP 1 THOUGHT: The user is expressing anger about a delayed order (123) and asking about a discount coupon if it doesn't arrive by tomorrow. The first step should be to check the status of order 123 to understand the situation before responding to the user's request for a coupon or providing a general response.
OBSERVATION: Order delayed by 3 days
STEP 2 THOUGHT: The user is angry about a delayed order and explicitly asks about a discount coupon if it's not received by tomorrow. The observation confirms a significant delay (3 days). Given this substantial delay, it's highly probable the order will not arrive by tomorrow, effectively meeting the user's condition for a coupon. Offering a coupon proactively addresses their request, acknowledges the inconvenience, and can help de-escalate their anger. 'fetch_order' is not needed as the delay is known. 'respond_to_user' is too general; 'offer_coupon' is a specific, actionable response to the user's direct request and the confirmed problem.

## Key Takeaways

- LLM stays inside a controlled loop
- Tools ground reasoning
- This pattern powers modern agent frameworks